# Purpose of Script
This script converts the .TXT, .DAT, and .ENCRYPTED files provided by the [FBI's Crime Data Explorer](https://cde.ucr.cjis.gov/LATEST/webapp/#/pages/downloads) into uniform .CSV files. 

## Initial Setup

In [4]:
import numpy as np
import pandas as pd

In [5]:
# Establish dictionary of region codes
regions = {'0':'Possessions', '1':'New England', '2':'Middle Atlantic', '3':'East North Central', '4':'West North Central',
           '5':'South Atlantic', '6':'East South Central', '7':'West South Central', '8':'Mountain', '9':'Pacific', ' ':''}
# Establish dictionary of offence type codes
offences = {'200':'Arson', '13A':'Aggravated Assault', '13B':'Simple Assault', '13C':'Intimidation', '510':'Bribery', '220':'Burglary',
            '250':'Forgery', '290':'Vandalism', '35A':'Drugs', '35B':'Drug Equipment', '270':'Embezzlement', '210':'Extortion', 
            '26A':'Swindle', '26B':'Credit Card/ATM Fraud', '26C':'Impersonation', '26D':'Welfare Fraud', '26E':'Wire Fraud',
            '39A':'Betting', '39B':'Promoting Gambling', '39C':'Gambling Equiptment', '39D':'Sports Tampering', '09A':'Murder',
            '09B':'Negligent Manslaughter', '09C':'Justifiable Homicide', '100':'Kidnapping', '23A':'Pick-Pocketing', '23B':'Purse Snatching',
            '23C':'Shoplifting', '23D':'Theft from Building', '23E':'Theft from Vending Machine', '23F':'Theft from Vehicle',
            '23G':'Theft of Vehicle Parts/Accessories', '23H':'Other Larceny', '240':'Vehicle Theft', '370':'Pornography', '40A':'Prostitution',
            '40B':'Promoting Prostitution', '120':'Robbery', '11A':'Rape', '11B':'Sodomy', '11C':'Sexual Assault with an Object', '11D':'Fondling',
            '36A':'Incest', '36B':'Statutory Rape', '280':'Receiving or Selling Stolen Property', '520':'Weapon Law Violations',
            '23*':'Theft', '40C':'Prostitution Related Unknown', '26F':'Non-Forcible Sexual Violation Unknown', '64A':'Unknown',
            '26G':'Fraud Uncategorized', '720':'Unknown', '101':'Unknown', '64B':'Unknown', '61A':'Unknown', '522':'Unknown'}
# Establish dictionary of location codes
locations = {'01':'Air/Bus/Train Terminal', '02':'Bank', '03':'Bar/Nightclub', '04':'House of Worship', '05':'Commercial/Office Building',
             '06':'Construction Site', '07':'Convenience Store', '08':'Department/Discount Store', '09':'Drug Store/Medical Building',
             '10':'Field/Woods', '11':'Government/Public Building', '12':'Grocery', '13':'Highway/Road/Alley', '14':'Hotel/Motel/Etc.',
             '15':'Jail/Prison', '16':'Lake/Waterway', '17':'Liquor Store', '18':'Parking Lot/Garage', '19':'Rental Storage Facility', 
             '20':'Residence/Home', '21':'Restaurant', '22':'School/College', '23':'Gas Station', '24':'Specialty Store', '25':'Other/Unknown',
             '37':'Abandoned Structure', '38':'Amusement Park', '39':'Arena/Stadium/Fairgrounds', '40':'ATM Separate From Bank',
             '41':'Auto Dealership', '42':'Campground', '44':'Daycare', '45':'Dock/Wharf/Freight Terminal', '46':'Farm Facility', 
             '47':'Casino', '48':'Industrial Site', '49':'Military Installation', '50':'Park/Playground', '51':'Rest Area',
             '52':'School - College/Univerisity', '53':'School - Elementary/Secondary', '54':'Shelter - Mission/Homeless', '55':'Shopping Mall',
             '56':'Tribal Lands', '57':'Community Center', '58':'Unknown'}
# Establish a dictionary of motivation codes
motivations = {'11':'Anti-White', '12':'Anti-Black', '13':'Anti-Native American', '14':'Anti-Asian', '15':'Anti-Multiple Races',
              '16':'Anti-Hawaiian/Pacific Islander', '21':'Anti-Jewish', '22':'Anti-Catholic', '23':'Anti-Protestant', '24':'Anti-Muslim',
              '25':'Anti-Other Religion', '26':'Anti-Multiple Religions', '27':'Anti-Atheist/Agnostic', '32':'Anti-Hispanic/Latino',
              '33':'Anti-Non-Hispanic/Latino', '41':'Anti-Gay (Male)', '42':'Anti-Lesbian', '43':'Anti-LGBT', '44':'Anti-Heterosexual',
              '45':'Anti-Bisexual', '51':'Anti-Physical Disability', '52':'Anti-Mental Disability', '61':'Anti-Male', '62':'Anti-Female',
              '71':'Anti-Transgender', '72':'Anti-Gender Nonconforming', '31':'Ethnicity Uncategorized', '28':'Anti-Mormon', 
              '29':'Anti-Jehovah\'s Witness', '81':'Anti-Eastern Orthodox', '82':'Anti-Other Christian', '83':'Anti-Buddhist',
              '84':'Anti-Hindu', '85':'Anti-Sikh', '99':'Unknown'}


## Convert Files to pandas DataFrames, then Export as CSV
Files were converted according to 
[the 2021 Record Description](../RawData/Hate%20Crime%20Help/2021%20Hate%20Crime%20Master%20File%20Record%20Description.pdf) provided with the raw data.

In [6]:
def convertToCSV(filename):

    # Initialize table to hold batch headers
    cols_bh = ['State', 'ORI', 'City', 'Region']
    bh = pd.DataFrame(columns=cols_bh)
    
    # Initialize table to hold incident reports
    cols_ir = ['ORI', 'Date', 'Victims', 'Offenders', 'Offence Type', 'Location', 'Motivation']
    ir = pd.DataFrame(columns=cols_ir)

    # Read file
    with open(filename, "r", encoding="utf-8") as file:
        for line in file:
            line = line.rstrip()
            # If this is a batch header row, add the info to the batch header DataFrame
            if line[0:2] == 'BH':
                new_row = pd.DataFrame({'State': [line[71:73]], 'ORI':[line[4:13]], 'City':[line[41:71].strip()], 'Region':[regions[line[75]]]})
                bh = pd.concat([bh, new_row], ignore_index = True)
            # If this is an incident report row, add the info to the incident report DataFrame
            elif line[0:2] == 'IR':
                new_row = pd.DataFrame({'ORI':[line[4:13]], 'Date':[line[25:33]], 'Victims':[line[35:38]], 'Offenders':[line[38:40]], 
                                        'Offence Type':[offences[line[41:44]]], 'Location':[locations[line[47:49]]],
                                        'Motivation':[motivations[line[49:51]]]})
                ir = pd.concat([ir, new_row], ignore_index = True)
        # Merge the DataFrames
        full_df = pd.merge(bh, ir, on='ORI', how='inner')
        # Save the merged DataFrame as a CSV
        full_df.to_csv(filename[:-3]+'csv', index=False)


In [7]:
convertToCSV("../RawData/HC2013.TXT")

In [8]:
convertToCSV("../RawData/HC2014.TXT")

In [9]:
convertToCSV("../RawData/HC2015.TXT")

In [10]:
convertToCSV("../RawData/HC2016.TXT")

In [11]:
convertToCSV("../RawData/HC2017.TXT")

In [12]:
convertToCSV("../RawData/HC2018.TXT")

In [13]:
convertToCSV("../RawData/HC2019.TXT")

In [14]:
convertToCSV("../RawData/HC2020.TXT")

In [15]:
convertToCSV("../RawData/HC2021.TXT")

In [16]:
convertToCSV("../RawData/HC2022.TXT")

In [17]:
convertToCSV("../RawData/HC2023.TXT")

In [18]:
convertToCSV("../RawData/HC2024.TXT")

In [19]:
convertToCSV("../RawData/HC2025PRELIM.TXT")